# Prepare Training Data

Custom tokenizer designed to correctly split text based on HTML tags and punctuation to avoid boundary errors during entity recognition.

In [ ]:
import re
import spacy
from spacy.tokenizer import Tokenizer
import spacy.util


prefix_re = re.compile(r'[.,;:?!(\[\'"</]')
suffix_re = re.compile(r'[.,;:?!)\]\'">■™]')
infix_re = re.compile(r'[-/+=&]')

def custom_tokenizer(nlp):
    tokenizer = Tokenizer(
        nlp.vocab,
        prefix_search=prefix_re.search,
        suffix_search=suffix_re.search,
        infix_finditer=infix_re.finditer
    )

    return tokenizer

Function to remove all HTML tags from the document and then realign the entity boundaries.

In [ ]:
from bs4 import BeautifulSoup

def clean_text_and_realign_entities(text, entities):
    # Step 1: Remove all HTML tags from the main text
    soup = BeautifulSoup(text, "html.parser")
    clean_text = soup.get_text()

    new_entities = []

    entity_position = {}

    # Get all the starting positions for each entity
    for entity in entities:
        entity_soup = BeautifulSoup(entity['text_span'], 'html.parser')
        clean_text_span = entity_soup.get_text().strip()

        if not clean_text_span:
            print('Empty span:', entity['text_span'])
            continue  # Skip empty spans

        entity_position[clean_text_span] = [match.start() for match in re.finditer(r'(?<!\w)'+re.escape(clean_text_span)+r'(?!\w)', clean_text)]

    # Remove starting position that are inside other entities
    for key in entity_position.keys():
        others = [other for other in entity_position.keys() if key in other and other not in key] # get all the keys containing key as substring
        starts = entity_position[key]

        for other in others:
            other_starts = entity_position[other]
            for index in starts:
                for other_index in other_starts:
                    if index >= other_index and index <= other_index + len(other):
                        starts.remove(index)
                
    # Update entity indexes
    for entity in entities:
        entity_soup = BeautifulSoup(entity['text_span'], 'html.parser')
        clean_text_span = entity_soup.get_text().strip()
        
        if len(entity_position[clean_text_span]) == 0:
            print(f"Warning: Span '{clean_text_span}' not found in text.")
            continue
        start_idx = entity_position[clean_text_span].pop(0)
        end_idx = start_idx + len(clean_text_span) - 1

        # Step 4: Update the entity with correct positions and cleaned span
        new_entity = {
            'start_idx': start_idx,
            'end_idx': end_idx,
            'text_span': clean_text_span,
            'label': entity['label']
        }
        new_entities.append(new_entity)

    return clean_text, new_entities


In [ ]:
def print_doc_tokens(title, doc, expected_text, label, start, end): # for debugging
    print('-'*60)
    print(title)
    
    tokens_span = []
    for token in doc:
        tokens_span.append((token, token.idx, token.idx + len(token.text)))
    
    print('Tokens -> ', tokens_span)
    print(f'Expected Entity: {expected_text} (Label: {label}, Start: {start}, End: {end})')

def get_spans(entities, doc, title):
    res = []
    for ent in entities:
        start = int(ent['start_idx'])
        end = int(ent['end_idx']) + 1
        label = ent['label']

        span = doc.char_span(start, end, label=label)
        if span:
            res.append(span)
        else:
            print(f'Bad span: {ent["text_span"]}')
            #print_doc_tokens(title, doc, ent['text_span'], label, start, end)

    # If two entities overlap, keep the one that covers more text
    filtered_ents = []
    for ent in sorted(res, key=lambda e: (e.start, -len(e.text))):  # Sort by start index, prefer longer entities
        if not any(ent.start < e.end and ent.end > e.start for e in filtered_ents):
            filtered_ents.append(ent)
    return filtered_ents

In [ ]:
def prepare_data(json_files, cleaning):
    db = DocBin()
    for json_file in json_files:
        print('#'*60)
        print('Parsing {}'.format(json_file))
        f = open(json_file)
        data = json.load(f)

        num_ents = 0
        num_parsed_ents = 0

        for article in data:
            title = data[article]['metadata']['title']
            abstract = data[article]['metadata']['abstract']
            entities = data[article]['entities']

            title_ents = [ent for ent in entities if ent['location'] == 'title']
            abstract_ents = [ent for ent in entities if ent['location'] == 'abstract']

            if cleaning:
                # Remove HTML tags and adjust entity indexes for title
                title, title_ents = clean_text_and_realign_entities(title, title_ents)
                
                # Remove HTML tags and adjust entity indexes for abstract
                abstract, abstract_ents = clean_text_and_realign_entities(abstract, abstract_ents)
    
            title_doc = nlp(title)
            abstract_doc = nlp(abstract)


            title_doc.ents = get_spans(title_ents, title_doc, title)
            abstract_doc.ents = get_spans(abstract_ents, abstract_doc, title)

            num_ents += len(title_ents) + len(abstract_ents)
            num_parsed_ents += len(title_doc.ents) + len(abstract_doc.ents)

            db.add(title_doc)
            db.add(abstract_doc)

        print(f'{num_parsed_ents}/{num_ents}')
        print(f'Articles: {len(data)}')
    return db

## Create .spacy files

In [ ]:
import json
from spacy.tokens import DocBin

train_json = [
    'gutbrainie2025/Annotations/Train/platinum_quality/json_format/train_platinum.json',
    'gutbrainie2025/Annotations/Train/gold_quality/json_format/train_gold.json',
    'gutbrainie2025/Annotations/Train/silver_quality/json_format/train_silver.json',
    'gutbrainie2025/Annotations/Train/bronze_quality/json_format/train_bronze.json'
]

dev_json = [
    'gutbrainie2025/Annotations/Dev/json_format/dev.json'
]

nlp = spacy.blank('en')

configs = ['base', 'no_tags', 'tok']
cleaning = False

for conf in configs:
    print('*'*60, conf)
    if conf == 'tok':
        nlp.tokenizer = custom_tokenizer(nlp)
    if conf == 'no_tags':
        cleaning = True
    train_db = prepare_data(train_json, cleaning)
    dev_db = prepare_data(dev_json, cleaning)
    train_db.to_disk(f'./train/{conf}.spacy')
    dev_db.to_disk(f'./dev/{conf}.spacy')

    cleaning = False

## Train the models

In [ ]:
!python -m spacy init fill-config base_config.cfg config.cfg
!python -m spacy train config.cfg --output ./out/base --paths.train ./train/base.spacy --paths.dev ./dev/base.spacy

In [ ]:
!python -m spacy init fill-config base_config.cfg config.cfg
!python -m spacy train config.cfg --output ./out/no_tags --paths.train ./train/no_tags.spacy --paths.dev ./dev/no_tags.spacy

In [ ]:
!python -m spacy train config-tok.cfg --code ./tokenizer.py --output ./out/tok --paths.train ./train/tok.spacy --paths.dev ./dev/tok.spacy

## Evaluate the models

In [ ]:
!python -m spacy evaluate out/base/model-best/ ./dev/base.spacy

In [ ]:
!python -m spacy evaluate out/tok/model-best/ ./dev/tok.spacy --code ./tokenizer.py

In [ ]:
!python -m spacy evaluate out/no_tags/model-best/ ./dev/no_tags.spacy